In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install --upgrade pip

In [2]:
!pip show langchain-groq

Name: langchain-groq
Version: 0.3.2
Summary: An integration package connecting Groq and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\Vangala_Angel\AppData\Roaming\Python\Python313\site-packages
Requires: groq, langchain-core
Required-by: 


In [3]:
!pip install --upgrade langchain-groq

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
import subprocess

package_name = "langchain-groq"

try:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", package_name])
    print(f"Successfully uninstalled {package_name}")
except subprocess.CalledProcessError as e:
    print(f"Error uninstalling {package_name}: {e}")
except FileNotFoundError:
    print("Error: pip command not found.")

Successfully uninstalled langchain-groq


In [2]:
!pip install langchain-groq

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_groq-0.3.2-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_groq-0.3.2-py3-none-any.whl (15 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install huggingface_hub transformers


Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from pptx import Presentation
from PyPDF2 import PdfReader
from langchain_core.documents import Document
from langchain.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
import shutil
import time
from langchain.llms.base import LLM
from langchain_groq import ChatGroq  
from langchain_mistralai.embeddings import MistralAIEmbeddings
from mistralai.client import MistralClient



In [2]:
folder_path = "Data"
chroma_path = "chroma"
def generate_data():
    """Loads data from PPT/PPTX and PDF files, splits it, and saves it to ChromaDB."""
    ppt_documents = load_powerpoint_from_folder(folder_path)
    pdf_documents = load_pdf_from_folder(folder_path)
    all_documents = ppt_documents + pdf_documents
    chunks = split_text(all_documents)
    save_to_chroma(chunks)

def load_powerpoint_from_folder(folder_path):
    """Loads text content from all PPT/PPTX files in a folder into Langchain Documents."""
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith((".ppt", ".pptx")):
            file_path = os.path.join(folder_path,filename)
            try:
                presentation = Presentation(file_path)
                text = ""
                for slide in presentation.slides:
                    for shape in slide.shapes:
                        if shape.has_text_frame:
                            for paragraph in shape.text_frame.paragraphs:
                                for run in paragraph.runs:
                                    text += run.text
                                text += "\n"  # Add a newline between paragraphs
                    text += "\n\n"  # Add a larger gap between slides
                metadata = {"source": file_path, "file_type": "PowerPoint"}
                documents.append(Document(page_content=text, metadata=metadata))
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    return documents

def load_pdf_from_folder(folder_path):
    """Loads text content from all PDF files in a folder into Langchain Documents."""
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            try:
                loader = PyPDFLoader(file_path)
                pdf_documents = loader.load()
                for doc in pdf_documents:
                    doc.metadata["source"] = file_path
                    doc.metadata["file_type"] = "PDF"
                    documents.append(doc)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    return documents

def split_text(documents:list[Document]):
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = text_splitter.split_documents(documents)
        print(f"split {len(documents)} documents into {len(chunks)} chunks.")
        return chunks
def save_to_chroma(chunks: list[Document]):
    embeddings = MistralAIEmbeddings()
    db = Chroma.from_documents(chunks, embeddings, persist_directory=chroma_path)
    db = None
    print(f"saved {len(chunks)} chunks to {chroma_path}.")
def query_data(query):
    # Initialize the embedding model
    embeddings = MistralAIEmbeddings()

    # Load the ChromaDB
    db = Chroma(persist_directory=chroma_path, embedding_function=embeddings)

    # Create a retriever
    retriever = db.as_retriever(search_kwargs={'k': 4})
    relevant_documents = retriever.invoke(query)
    context = "\n\n".join([doc.page_content + f" (Source: {doc.metadata.get('file_type', 'unknown')})" for doc in relevant_documents])

    prompt = f"""You will try to answer the following question based on the provided documents.
Prioritize information found directly within the content of PowerPoint presentations or PDF documents.
If the answer is clearly and sufficiently present in the provided document content, use that information to answer.
Cite the source of your information by mentioning "(from PowerPoint)" or "(from PDF)".

If, after reviewing the document content, you cannot find a direct and complete answer, or if the information is insufficient, then you can use your general knowledge to provide a more comprehensive answer. In this case, do not cite a specific document.

Question: {query}

Document Content:
{context}

Answer: """

    try:
        groq_api_key = os.environ.get("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("GROQ_API_KEY environment variable not set.")

        llm = ChatGroq(api_key=groq_api_key, model_name="llama3-70b-8192")
        answer = llm.invoke(prompt)  # This works for ChatGroq!
        print("\nAnswer for query: ")
        print(answer.content)

    except ImportError:
        print("\nLangchain Groq library not found. Please install it using 'pip install langchain-groq'.")
    except ValueError as e:
        print(f"\nError: {e}")
    except Exception as e:
        print(f"\nError during Groq API call: {e}")
    finally:
        del db

generate_data()

split 298 documents into 579 chunks.


C:\Users\Vangala_Angel\AppData\Roaming\Python\Python313\site-packages\langchain_mistralai\embeddings.py:181: UserWarning: Could not download mistral tokenizer from Huggingface for calculating batch sizes. Set a Huggingface token via the HF_TOKEN environment variable to download the real tokenizer. Falling back to a dummy tokenizer that uses `len()`.
  warnings.warn(


saved 579 chunks to chroma.


In [3]:
def main(query):
    query_data(query)

In [ ]:
while True:
    query=input("Enter your Question: ")
    if query.lower()=="quit":
        print("Hope this was helpful.")
        print("See you again soon.")
        break
    else:
        main(query)

Enter your Question:  what is ai



Answer for query: 
According to the provided documents, AI can be defined as:

"The automation of activities that we associate with human thinking (e.g., decision-making, learning…)" (from PowerPoint)

or

"The study of how to make computers make things which at the moment people do better." (from PowerPoint)

or

"The branch of computer science that is concerned with the automation of intelligent behavior." (from PowerPoint)

In general, AI is the field of study that seeks to explain and emulate intelligent behavior in terms of computational processes, and it involves the development of machines that can perform functions that require intelligence when performed by humans.
